Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


In [2]:
import kagglehub

path = kagglehub.competition_download("playground-series-s6e9")

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/playground-series-s6e9


In [3]:
train = pd.read_csv("../input/competitions/playground-series-s6e9/train.csv")
test = pd.read_csv("../input/competitions/playground-series-s6e9/test.csv")
sample_submission = pd.read_csv("../input/competitions/playground-series-s6e9/sample_submission.csv")

# 関数の定義

## ハイパーパラメータ

- `max_depth` : 3 が底。2 ～ 3 個の変数の組み合わせが限界、それ以上は過学習する。
- `num_leaves` : `max_depth` に対応。

In [4]:
def train_lgb(train_df, test_df, feature_cols, target_col="Will_Buy_EV", seed=0):
    
    """指定された特徴量とシード値で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    # 1. 目的変数の変換
    y_train = (
        (train_df[target_col] == "Yes").astype(int).values
    )  # NumPy配列にして高速化

    # カテゴリ型への一括変換
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()

    for col in feature_cols:
        if X_train_df[col].dtype == "object":
            X_train_df[col] = X_train_df[col].astype("category")
            X_test_df[col] = X_test_df[col].astype("category")

    X_train_df = X_train_df.reset_index(drop=True)

    # 💡 引数で受け取った seed を KFold の random_state に適用
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)

    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "max_depth": 3,
        "num_leaves": 8,
        "learning_rate": 0.1,  # 仕上げ時は 0.03 等に変更可
        "colsample_bytree": 0.8,
        "n_jobs": -1,
        "random_state": seed,  # 💡 引数で受け取った seed を LightGBM に適用
        "verbose": -1,
        "enable_categorical": True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    test_preds = np.zeros(len(X_test_df))  # 各Foldのテスト予測を累積する用

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train_df, y_train)
    ):
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(
            X_va, y_va, reference=lgb_train, free_raw_data=True
        )

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval],
            num_boost_round=3000,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
        )

        best_iter = model.best_iteration
        best_score = model.best_score["valid_0"]["auc"]
        print(
            f"Fold {fold + 1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}"
        )

        # 検証データの予測
        oof_train[valid_idx] = model.predict(
            X_va, num_iteration=model.best_iteration
        )

        # テストデータの予測（Fold ごとに加算して後で 5 で割る）
        test_preds += (
            model.predict(X_test_df, num_iteration=model.best_iteration) / 5
        )

        models.append(model)

    score = roc_auc_score(y_train, oof_train)
    print(f"OOF ROC-AUC Score: {score:.5f}")

    # 💡 5 Fold 分の平均テスト予測値 (test_preds) をそのまま返す
    return oof_train, test_preds, models

# EDA

## binning

### commute_bin

通勤距離のビン分け

In [5]:
# 通勤距離のビン分け（範囲の定義）
# 0〜8km（5kmの大量データ含む）、8〜20km、20〜60km（中距離）、60〜80km（長距離）、80km〜（超長距離）
bins = [0, 8, 20, 60, 80, float('inf')]
labels = ['short_5km_zone', 'short_mid', 'mid_distance', 'long_distance', 'super_long_distance']

# pd.cut でカテゴリ変数化（LGBMで直接扱えるように category 型にキャスト）
train['commute_bin'] = pd.cut(train['Daily_Commute_km'], bins=bins, labels=labels).astype('category')
test['commute_bin'] = pd.cut(test['Daily_Commute_km'], bins=bins, labels=labels).astype('category')

### income_bin

年収のビン分け

In [6]:
income_bins = [0, 30000, 60000, 100000, 150000, float('inf')]
income_labels = ['income_min_30k', 'low_income', 'mid_income', 'high_income', 'super_high_income']

train['income_bin'] = pd.cut(train['Annual_Income_USD'], bins=income_bins, labels=income_labels).astype('category')
test['income_bin'] = pd.cut(test['Annual_Income_USD'], bins=income_bins, labels=income_labels).astype('category')

## env_x_anxiety_cat

環境意識 × 不安度（心理的バランス）

In [7]:
# Range_Anxiety_Level の数値化
anxiety_map = {'Low': 1, 'Medium': 2, 'High': 3}

train['range_anxiety_num'] = train['Range_Anxiety_Level'].map(anxiety_map)
test['range_anxiety_num'] = test['Range_Anxiety_Level'].map(anxiety_map)

In [8]:
# "5.0_Low", "1.0_High" のような 15 通りの組み合わせを作成
train['env_x_anxiety_cat'] = (train['Environmental_Concern_Level'].astype(str) + '_' + train['Range_Anxiety_Level'].astype(str)).astype('category')
test['env_x_anxiety_cat'] = (test['Environmental_Concern_Level'].astype(str) + '_' + test['Range_Anxiety_Level'].astype(str)).astype('category')

## Charging_Convenience

`Charging_Stations_Near_Home` + `Charging_Stations_Near_Work`

In [9]:
train["Charging_Convenience"] = train["Charging_Stations_Near_Home"] + train["Charging_Stations_Near_Work"]
test["Charging_Convenience"] = test["Charging_Stations_Near_Home"] + test["Charging_Stations_Near_Work"]

## income_per_car

1 台あたりの資金的余裕

In [10]:
# 0 除算を防ぐため np.where を使用（0台の場合はそのまま年収、または np.nan / 特定値に）
train['income_per_car'] = train['Annual_Income_USD'] / np.where(
    train['Number_of_Cars_Owned'] == 0, 1, train['Number_of_Cars_Owned']
)
test['income_per_car'] = test['Annual_Income_USD'] / np.where(
    test['Number_of_Cars_Owned'] == 0, 1, test['Number_of_Cars_Owned']
)

## gender_x_city

都市型 × 性別の生活スタイル

In [11]:
train['gender_x_city'] = (
    train['Gender'].astype(str) + '_' + train['City_Type'].astype(str)
).astype('category')
test['gender_x_city'] = (
    test['Gender'].astype(str) + '_' + test['City_Type'].astype(str)
).astype('category')

## gender_x_car

現在の車種 × 性別の生活スタイル

In [12]:
train['gender_x_car'] = (
    train['Gender'].astype(str) + '_' + train['Current_Car_Type'].astype(str)
).astype('category')
test['gender_x_car'] = (
    test['Gender'].astype(str) + '_' + test['Current_Car_Type'].astype(str)
).astype('category')

## Buy_score

[https://www.kaggle.com/code/itzzomkar/the-complete-original-generator-script](http://)

In [13]:
def calc_buy_score(df):
    # 1. 年収
    score = (df['Annual_Income_USD'] / 250000.0) * 0.3

    # 2. 環境意識
    score += (df['Environmental_Concern_Level'] / 5.0) * 0.3

    # 3. 補助金
    score += np.where(df['Subsidy_Available'] == "Yes", 0.2, 0.0)

    # 4. 航続距離不安
    score -= np.where(df['Range_Anxiety_Level'] == "High", 0.3, 0.0)
    score -= np.where(df['Range_Anxiety_Level'] == "Medium", 0.1, 0.0)

    return score


train['Buy_score'] = calc_buy_score(train)
test['Buy_score'] = calc_buy_score(test)

### Score_Economic

In [14]:
train['Score_Economic'] = (
    0.3 * (train['Annual_Income_USD'] / 250000)
    + 0.2 * (train['Subsidy_Available'] == 'Yes').astype(int)
)

test['Score_Economic'] = (
    0.3 * (test['Annual_Income_USD'] / 250000)
    + 0.2 * (test['Subsidy_Available'] == 'Yes').astype(int)
)

### Score_Psychological

In [15]:
train['Score_Psychological'] = (
    0.3 * (train['Environmental_Concern_Level'] / 5.0)
    - 0.1 * (train['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 0.3 * (train['Range_Anxiety_Level'] == 'High').astype(int)
)

test['Score_Psychological'] = (
    0.3 * (test['Environmental_Concern_Level'] / 5.0)
    - 0.1 * (test['Range_Anxiety_Level'] == 'Medium').astype(int)
    - 0.3 * (test['Range_Anxiety_Level'] == 'High').astype(int)
)

# スコアリング特徴量

全体購入率 17.5% を基準に閾値を決める。
25% 以上（全体購入率の 1.4 倍以上）から、購入率が高いと設定。

In [16]:
# 1. プラス要素のカウント（購入率 25% 以上の条件）
train['Pos_score'] = 0
test['Pos_score'] = 0

# 重み調整用の変数
num1 = 1
num2 = 1.5
num3 = 2

# ゆるいプラス条件（購入率 25 ~ 35%） -> +1 点
train['Pos_score'] += (train['Subsidy_Available'] == 'Yes').astype(int) * num1
test['Pos_score'] += (test['Subsidy_Available'] == 'Yes').astype(int) * num1
train['Pos_score'] += (train['Environmental_Concern_Level'] == 4.0).astype(int) * num1
test['Pos_score'] += (test['Environmental_Concern_Level'] == 4.0).astype(int) * num1
train['Pos_score'] += (train['commute_bin'] == 'short_mid').astype(int) * num1
test['Pos_score'] += (test['commute_bin'] == 'short_mid').astype(int) * num1
train['Pos_score'] += (train['income_bin'] == 'high_income').astype(int) * num1
test['Pos_score'] += (test['income_bin'] == 'high_income').astype(int) * num1
train['Pos_score'] += (train['env_x_anxiety_cat'] == '4.0_Low').astype(int) * num1
test['Pos_score'] += (test['env_x_anxiety_cat'] == '4.0_Low').astype(int) * num1

# 強力なプラス条件（購入率 35 ~ 50%） -> +2 点
train['Pos_score'] += (train['income_bin'] == 'super_high_income').astype(int) * num2
test['Pos_score'] += (test['income_bin'] == 'super_high_income').astype(int) * num2

# 非常に強力なプラス条件（購入率 50% 以上） -> +3 点
train['Pos_score'] += (train['Environmental_Concern_Level'] == 5.0).astype(int) * num3
test['Pos_score'] += (test['Environmental_Concern_Level'] == 5.0).astype(int) * num3
train['Pos_score'] += (train['env_x_anxiety_cat'] == '5.0_Low').astype(int) * num3
test['Pos_score'] += (test['env_x_anxiety_cat'] == '5.0_Low').astype(int) * num3

In [17]:
# 2. マイナス要素のカウント（購入率が極端に低い 10% 未満の条件）
train['Neg_score'] = 0
test['Neg_score'] = 0


# ゆるいマイナス条件（購入率 7.5 ~ 10% 未満）
train['Neg_score'] += (train['income_bin'] == 'low_income').astype(int) * num1
test['Neg_score'] += (test['income_bin'] == 'low_income').astype(int) * num1


# 強力なマイナス条件（購入率 2.5 ~ 7.5% 未満）
train['Neg_score'] += (train['range_anxiety_num'] >= 2.0).astype(int) * num2
test['Neg_score'] += (test['range_anxiety_num'] >= 2.0).astype(int) * num2
train['Neg_score'] += (train['income_bin'] == 'income_min_30k').astype(int) * num2
test['Neg_score'] += (test['income_bin'] == 'income_min_30k').astype(int) * num2


# 非常に強力なマイナス条件（購入率 2.5% 未満）
train['Neg_score'] += (train['Subsidy_Available'] == 'No').astype(int) * num3
test['Neg_score'] += (test['Subsidy_Available'] == 'No').astype(int) * num3
train['Neg_score'] += (train['Environmental_Concern_Level'] <= 2.0).astype(int) * num3
test['Neg_score'] += (test['Environmental_Concern_Level'] <= 2.0).astype(int) * num3

In [18]:
# 3. プラス要素とマイナス要素の差分
# 学習に使うのはこれだけ、Pos_score と Neg_score はバリアンスが高いので中間変数としてのみ活用
train['Total_score'] = train['Pos_score'] - train['Neg_score']
test['Total_score'] = test['Pos_score'] - test['Neg_score']

# クラスタリング

In [19]:
binary_map = {'Yes': 1, 'No': 0}

# train の変換
train['Home_Charging_Possible_num'] = (
    train['Home_Charging_Possible'].map(binary_map).astype(int)
)
train['Subsidy_Available_num'] = (
    train['Subsidy_Available'].map(binary_map).astype(int)
)

# test の変換
test['Home_Charging_Possible_num'] = (
    test['Home_Charging_Possible'].map(binary_map).astype(int)
)
test['Subsidy_Available_num'] = (
    test['Subsidy_Available'].map(binary_map).astype(int)
)

In [20]:
# 「都市 & 充電インフラ」の環境特徴量を定義
cluster_features = [
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Charging_Convenience",
    "Home_Charging_Possible_num",
]

# スケーリング（StandardScaler で標準化: KMeans に必須）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train[cluster_features])
X_test_scaled = scaler.transform(test[cluster_features])

# 指定したクラスタ数でクラスタリング実行
k = 4
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)

train["infrastructure_cluster"] = kmeans.fit_predict(X_train_scaled)
test["infrastructure_cluster"] = kmeans.predict(X_test_scaled)

# LightGBM で扱えるように category 型に変換
train["infrastructure_cluster"] = train["infrastructure_cluster"].astype(
    "category"
)

test["infrastructure_cluster"] = test["infrastructure_cluster"].astype(
    "category"
)

In [21]:
# 経済系の特徴量を定義
# 3 つのクラスタ中心点からの距離（3 次元の連続値特徴量） に変換してアプローチ
cluster_features = [
    'Annual_Income_USD',
    'Subsidy_Available_num',
    'income_per_car',
    'Score_Economic',
]

scaler = StandardScaler()
X_train_econ = scaler.fit_transform(train[cluster_features])
X_test_econ = scaler.transform(test[cluster_features])

# KMeans を初期化して学習 (fit)
k = 3
kmeans_econ = KMeans(n_clusters=k, random_state=0, n_init=10)
kmeans_econ.fit(X_train_econ)

# 各クラスタ中心点からの距離 (transform) を算出
train_econ_dist = kmeans_econ.transform(X_train_econ)
test_econ_dist = kmeans_econ.transform(X_test_econ)

# 連続値特徴量（dist_to_econ_cluster_0 ～ 2）として train/test に追加
for i in range(k):
    train[f"dist_to_econ_cluster_{i}"] = train_econ_dist[:, i]
    test[f"dist_to_econ_cluster_{i}"] = test_econ_dist[:, i]

In [22]:
# スコア関連の特徴量を定義
cluster_features = [
    'Buy_score', 
    'Score_Economic', 
    'Score_Psychological',
]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train[cluster_features])
X_test_scaled = scaler.transform(test[cluster_features])

k = 3
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)

train['score_cluster'] = kmeans.fit_predict(X_train_scaled)
test['score_cluster'] = kmeans.predict(X_test_scaled)

train['score_cluster'] = train['score_cluster'].astype('category')
test['score_cluster'] = test['score_cluster'].astype('category')

# エラー分析の反映

In [23]:
def create_gap_flag_features(df):
    # --- 条件の定義 ---
    # 1. 真に絶望的なインフラ層
    strict_low_infra = (df["Home_Charging_Possible"] == "No") & (
        df["Charging_Convenience"] <= 2
    )
    cond1 = (df["Environmental_Concern_Level"] == 5.0) & strict_low_infra

    # 2. 意識は低いが補助金/経済面で買う層
    high_economic_condition = (df["Subsidy_Available"] == "Yes") | (
        df["Annual_Income_USD"] >= 100000.0
    )
    cond2 = (df["Environmental_Concern_Level"] <= 2.0) & high_economic_condition

    # --- 個別フラグ（0 / 1） ---
    df["flag_high_concern_low_infra_strict"] = cond1.astype(int)
    df["flag_low_concern_high_subsidy"] = cond2.astype(int)

    # --- 1つのカテゴリ変数（gap_category）にまとめる ---
    # 0: 通常層, 1: インフラ絶望層, 2: 経済後押し層
    gap_cat = np.zeros(len(df), dtype=int)
    gap_cat[cond1] = 1
    gap_cat[cond2] = 2

    # LightGBM がカテゴリ特徴量として正しく処理できるように 'category' 型に変換
    df["gap_category"] = pd.Series(gap_cat, index=df.index).astype("category")

    return df


train = create_gap_flag_features(train)
test = create_gap_flag_features(test)

In [24]:
# 「補助金あり & 自宅充電可能 & 航続距離不安 Low」の安定的購入層フラグ
train["flag_safe_buyer"] = (
    (train["Subsidy_Available"] == "Yes")
    & (train["Home_Charging_Possible"] == "Yes")
    & (train["Range_Anxiety_Level"] == "Low")
).astype(int)

test["flag_safe_buyer"] = (
    (test["Subsidy_Available"] == "Yes")
    & (test["Home_Charging_Possible"] == "Yes")
    & (test["Range_Anxiety_Level"] == "Low")
).astype(int)

# 実行

In [25]:
# 1. 不要な列の除外（特徴量の選択）
features = [
    col
    for col in train.columns
    if col not in [
    "id",
    "Will_Buy_EV",
    "Range_Anxiety_Level",
    'range_anxiety_num',
    'Home_Charging_Possible_num',
    'Subsidy_Available_num',
    'City_Type',
    'Current_Car_Type',
    'Pos_score',
    'Neg_score',
    'income_bin_x_subsidy',
    ]
]

# 2. 【型変換の一括適用】Daily_Commute_kmの100倍整数化と、各列の型軽量化
train_copy = train.copy()
test_copy = test.copy()

# ① Daily_Commute_km を100倍して丸める
if 'Daily_Commute_km' in features:
    train_copy['Daily_Commute_km'] = (train_copy['Daily_Commute_km'] * 100).round()
    test_copy['Daily_Commute_km'] = (test_copy['Daily_Commute_km'] * 100).round()

# ② 辞書型を用いて一括型キャスト
type_mapping = {}
for col in features:
    if train_copy[col].dtype == 'object':
        type_mapping[col] = 'category'
    elif col in ['Annual_Income_USD', 'Daily_Commute_km']:
        type_mapping[col] = 'int32'
    elif col == 'Environmental_Concern_Level':
        type_mapping[col] = 'int16'

train_copy = train_copy.astype(type_mapping)
test_copy = test_copy.astype(type_mapping)


# 3. 余計な列を完全に排除した「純粋な軽量DataFrame」を孤立させて作成
pure_train = train_copy[features + ["Will_Buy_EV"]].copy()
pure_test = test_copy[features].copy()

In [26]:
%%time

seeds = [0, 42, 123, 2024, 9999]

# 予測結果を蓄積する配列を初期化
oof_preds_all = np.zeros(len(pure_train))
test_preds_all = np.zeros(len(pure_test))

# シードごとにモデルを学習させて平均をとる
for i, seed in enumerate(seeds):
    print(
        f"\n==================== Running Seed {seed} ({i+1}/{len(seeds)}) ===================="
    )

    # ※ train_lgb が seed 引数を受け取れるように定義を調整
    oof_pred, test_pred, _ = train_lgb(
        pure_train, pure_test, features, seed=seed
    )

    oof_preds_all += oof_pred / len(seeds)
    test_preds_all += test_pred / len(seeds)

# 5 つのシードを平均した最終的な OOF ROC-AUC スコアを計算
final_score = roc_auc_score(
    (pure_train["Will_Buy_EV"] == "Yes").astype(int), oof_preds_all
)
print(f"\n==========================================")
print(f"Final Seed Averaged OOF ROC-AUC Score: {final_score:.5f}")
print(f"==========================================")


==================== Running Seed 0 (1/5) ====================
Fold 1 | Best Iteration: 1445 | Best AUC: 0.94168
Fold 2 | Best Iteration: 1252 | Best AUC: 0.94188
Fold 3 | Best Iteration: 1604 | Best AUC: 0.94245
Fold 4 | Best Iteration: 1502 | Best AUC: 0.94174
Fold 5 | Best Iteration: 1737 | Best AUC: 0.94412
OOF ROC-AUC Score: 0.94236

==================== Running Seed 42 (2/5) ====================
Fold 1 | Best Iteration: 1136 | Best AUC: 0.94255
Fold 2 | Best Iteration:  780 | Best AUC: 0.94119
Fold 3 | Best Iteration:  777 | Best AUC: 0.94291
Fold 4 | Best Iteration: 1586 | Best AUC: 0.94210
Fold 5 | Best Iteration: 1500 | Best AUC: 0.94283
OOF ROC-AUC Score: 0.94230

==================== Running Seed 123 (3/5) ====================
Fold 1 | Best Iteration: 1055 | Best AUC: 0.94261
Fold 2 | Best Iteration:  705 | Best AUC: 0.94147
Fold 3 | Best Iteration: 1467 | Best AUC: 0.94285
Fold 4 | Best Iteration: 1123 | Best AUC: 0.94147
Fold 5 | Best Iteration: 1443 | Best AUC: 0.94304
O

# 提出

In [27]:
submission = sample_submission.copy()
submission["Will_Buy_EV"] = test_preds_all

submission.to_csv("submission.csv", index=False)
print("submission.csv を作成しました！")


# ファイルの内容・件数・欠損値のチェック
print("\n--- 先頭 5 行の確認 ---")
print(submission.head())

print("\n--- データ件数の確認 ---")
print(
    f"作成した件数: {len(submission)} / 元の件数: {len(sample_submission)}"
)

print("\n--- 欠損値（Null）の有無チェック ---")
print(f"欠損値の数: {submission['Will_Buy_EV'].isnull().sum()}")

print("\n--- 予測値（確率）の基本統計量 ---")
print(submission["Will_Buy_EV"].describe())

submission.csv を作成しました！

--- 先頭 5 行の確認 ---
       id  Will_Buy_EV
0  668665     0.011360
1  668666     0.020475
2  668667     0.004386
3  668668     0.002828
4  668669     0.022560

--- データ件数の確認 ---
作成した件数: 286571 / 元の件数: 286571

--- 欠損値（Null）の有無チェック ---
欠損値の数: 0

--- 予測値（確率）の基本統計量 ---
count    2.865710e+05
mean     1.747023e-01
std      2.701138e-01
min      1.718275e-07
25%      2.202847e-03
50%      1.801223e-02
75%      2.684017e-01
max      9.975807e-01
Name: Will_Buy_EV, dtype: float64


In [28]:
# sub = sample_submission.copy()
# sub["Will_Buy_EV"] = y_preds

# # CSVファイルとして保存
# sub.to_csv("submission.csv", index=False)
# print("\nsubmission.csv has been saved!")